<a href="https://colab.research.google.com/github/SehrishbAsghar/FlyRank_ML_Internship_Sehrish/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

In [6]:
rel = "hf://datasets/FlyRank/internship-warehouse"
test = con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')").df()
print(test)

   count_star()
0       9841378


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one pseudonymized content item, for one pseudonymized client, on one
report date.

Source table: `fact_content_daily_performance`

Time window:
`month=2026-03`

A mid-panel month, deliberately avoiding the sealed final month,
June 2026, which is reserved as the test window for any past-future label.


In [8]:
schema_check = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1").df()
print(schema_check)

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [9]:
rel = "hf://datasets/FlyRank/internship-warehouse"

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_combos
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_combos
0     9841378        9841378


**Verified above:**

Total_rows (9,841,378) exactly equals unique (client_hash_id,
content_hash_id, report_date) combinations (9,841,378)

Confirming the grain is correct, no duplicate rows.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (inputs to my scoring model):
`gsc_impressions`, `gsc_clicks`  needed to compute CTR and check volume/reliability
- `gsc_avg_position` position tier, the main driver of expected CTR
- `content_type` (joined from `dim_content`) grouping variable for expected CTR
- `ga4_engaged_sessions`, `ga4_total_engagement_sec` the "engagement" half of my lane; per the lane guide, "sessions and engagement context" is a listed good feature for CTR/Engagement Opportunity Scoring
- `scroll_events` engagement signal, also explicitly relevant per the guide

**Label / proxy** (what I'm scoring):
- Computed CTR gap: `(gsc_clicks / gsc_impressions)` compared against the group-expected CTR for that `position_tier` + `content_type`. Not a raw column derived from `gsc_clicks` and `gsc_impressions` together.

**Context** (not fed to the model, but needed to interpret/filter results):
- `client_hash_id`, `content_hash_id`, `report_date` identifiers, not predictive signal
- `client_has_gsc`, `gsc_data_available`used to filter for valid rows, not as model input
- `month` used for slicing the time window, not a feature.

**Excluded:**
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`,
  `sessions_paid`, `sessions_ai`, and all `ai_*` columns( `ai_chatgpt`, `ai_claude`,`ai_other`, etc.) These are traffic-source breakdowns. Mixing channel-attribution data into a CTR/engagement score would blur what's actually driving the opportunity signal.
- `gsc_sum_position` redundant with `gsc_avg_position` (sum is just avg x count), no added information.

In [10]:
rel = "hf://datasets/FlyRank/internship-warehouse"

verify = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_impressions,
        SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS null_clicks,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS null_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(verify)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   min_date   max_date  null_impressions  null_clicks  \
0    9841378 2026-03-01 2026-03-31               0.0          0.0   

   null_position  
0      6230317.0  


In [11]:
rel = "hf://datasets/FlyRank/internship-warehouse"

availability_check = con.sql(f"""
    SELECT
        gsc_data_available,
        COUNT(*) AS row_count,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS null_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY gsc_data_available
""").df()

print(availability_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   gsc_data_available  row_count  null_position
0               False    6230317      6230317.0
1                True    3611061            0.0


**Availability check:**

Filtering on `gsc_data_available IS TRUE` removes exactly
the rows with missing `gsc_avg_position`, 6,230,317 rows (63.3%) are dropped because GSC data wasn't available for that client/day, and the remaining
3611,061 rows (36.7%) have complete GSC data with zero nulls in impressions, clicks, or position. This single filter is both the availability check and the missing-value fix for this table.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every contract claim above is checked against a real query below.

# **Grain check:Grain check**

In [12]:
rel = "hf://datasets/FlyRank/internship-warehouse"

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_combos
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_combos
0     9841378        9841378


Grain confirmed, no duplicates.

# **Query 2: Counts, date span, missing values**

In [13]:
verify = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_impressions,
        SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS null_clicks,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS null_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(verify)

   row_count   min_date   max_date  null_impressions  null_clicks  \
0    9841378 2026-03-01 2026-03-31               0.0          0.0   

   null_position  
0      6230317.0  


zero nulls in impressions/clicks, but 6,230,317 nulls in gsc_avg_position

# **Query 3: Availability check (IS TRUE)**

In [14]:
availability_check = con.sql(f"""
    SELECT
        gsc_data_available,
        COUNT(*) AS row_count,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS null_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY gsc_data_available
""").df()
print(availability_check)

   gsc_data_available  row_count  null_position
0                True    3611061            0.0


The IS TRUE filter resolves the missing-position problem completely.

All three contract claims are now backed by queries:

Grain is exact (no duplicates)


The time window is a complete month, and

Filtering on `gsc_data_available IS TRUE` cleanly separates usable rows (3,611,061) from rows with no real GSC data (6,230,317, all null-position).

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Availability isn't random.**

63.3% of March rows have `gsc_data_available = False` this isn't missing-at-random noise, it's structural (clients whose GSC tracking wasn't active that day).

Filtering to `IS TRUE` gives clean data, but it also means, I am only ever seeing the subset of client-days where tracking existed, not a representative sample of all content performance,just content performance where GSC was already reporting.

**Unbalanced panel across clients.**

Per the lane guide, different clients have different amounts of tracking history (`gsc_data_start`/`ga4_data_start` vary), and only 9 of 70 clients have 12+ months of history. A single mid-panel month
like March 2026 will include some clients mid-history and possibly miss others whose tracking started later so, patterns I find in this month may not generalize evenly across all clients in the warehouse.

**GSC-only early rows / GA4 gaps.**

Rows before a client's GA4 start will have search data but no engagement data(`ga4_data_available = FALSE`), which matters directly for my "Engagement" half of the lane. Some content items may look like they have no engagement signal simply because GA4 wasn't tracking yet, not because engagement was actually zero.

**This data can't explain *why*, only *what*.**

Even a clean CTR gap only shows
that a page underperforms its position/type peers it can't say whether that's a bad title, wrong search intent, seasonality, or consolidation with a sibling page. Any "opportunity" flag is a prioritization signal for human review, not a diagnosis.

**One month can't show trend or persistence.**

Since I am only using `month=2026-03` (a single mid-panel month, deliberately not the sealed final month), I can observe a snapshot of CTR vs. Expected, but I can't yet tell if a low-CTR page is a persistent pattern or a one-month blip that would need a multi-month window,
which is out of scope for this contract.


## Self-check

Before you submit, confirm each line honestly:

- Every section above is filled — markdown thinking AND the code that backs it
- The notebook runs top to bottom with no errors (Runtime → Run all)
- No client names, URLs, or private queries anywhere
- My claims use careful words: observed, measured, directional, decision-support
- Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.